In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as psf
from pyspark.sql import DataFrame
from pyspark.sql.window import Window

spark: SparkSession = (
    SparkSession.builder.config(
        "spark.driver.extraJavaOptions", "-Djava.security.manager=allow"
    )
    .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow")
    .getOrCreate()
)

In [ ]:
def clean(df: DataFrame) -> DataFrame:
    return df.withColumn("name", psf.initcap("name"))


df_batch_measurements = spark.read.parquet("../data/batch_measurements.parquet")
df_dim_vet = spark.read.parquet("../data/dim_vet.parquet")
df_visitors = spark.read.parquet("../data/visitors.parquet")

How many times has vet practice Digital Wildlife Care diagnosed a polar bear as sick?


In [ ]:
practice = "Digital Wildlife Care"

(
    df_batch_measurements.transform(clean)
    .join(df_dim_vet, on="vet", how="left")
    .filter(psf.col("practice") == practice)
    .filter(psf.col("vet_health_check") == "SICK")
    .count()
)

Which vet(s) has/have never seen a weight above the 99.9th percentile? (Hint: the keyword to search for is `quantile`.)


In [ ]:
bigboi = df_batch_measurements.transform(clean).select(
    psf.percentile("weight", 0.999).alias("bigboi")
)

df_vet_seen_bigboi = (
    df_batch_measurements.transform(clean)
    .join(df_dim_vet, on="vet", how="left")
    .join(bigboi, how="cross")
    .filter(psf.col("weight") > psf.col("bigboi"))
    .select("vet")
    .distinct()
)

df_dim_vet.join(df_vet_seen_bigboi, on="vet", how="left_anti").show(10, False)

Which vet was the least consistent in name capitalization?


In [ ]:
consistent_name = psf.when(psf.col("name") == psf.initcap("name"), 1).otherwise(0)

(
    df_batch_measurements.join(
        df_dim_vet.withColumnRenamed("name", "name_vet"), on="vet", how="left"
    )
    .groupby("name_vet")
    .agg(
        psf.sum(consistent_name).alias("nr_consistent"),
        psf.count("*").alias("nr_total"),
    )
    .withColumn("%", psf.col("nr_consistent") / psf.col("nr_total") * 100)
    .sort("%")
    .show(10, False)
)

Which day had the lowest bear-to-visitor ratio? A bear can be seen if the vet determines the bear is healthy. (Hint: while joins require an exact match, this question is most easily solved using a non-exact match.)


In [ ]:
df_healthy_bears = (
    df_batch_measurements.transform(clean)
    .withColumn("day", psf.to_date("timestamp"))
    .filter(psf.col("vet_health_check") == "HEALTHY")
    .groupBy("day")
    .agg(psf.count("*").alias("healthy_bears"))
)

window = Window.orderBy("day")

(
    df_visitors.withColumn("day", psf.to_date("timestamp"))
    .join(df_healthy_bears, on="day", how="left")
    .withColumn(
        "healthy_bears", psf.last("healthy_bears", ignorenulls=True).over(window)
    )
    .filter(psf.col("healthy_bears").isNotNull())
    .withColumn("bear_to_visitor_ratio", psf.col("healthy_bears") / psf.col("visitors"))
    .sort("bear_to_visitor_ratio")
    .show(10, False)
)